In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import os
from fof8_core.loader import FOF8Loader

# 1. Setup absolute paths
REPO_ROOT = "/workspaces/fof8-scout"
FEATURES_PATH = os.path.join(REPO_ROOT, "fof8-ml/data/processed/features.parquet")
RAW_PATH = os.path.join(REPO_ROOT, "fof8-gen/data/raw")
LEAGUE_NAME = "DRAFT005"
RIGHT_CENSOR_BUFFER = 20
TEST_SPLIT_PCT = 0.2
TARGET_COL = "DPO"

# 2. Discover simulation timeline
loader = FOF8Loader(base_path=RAW_PATH, league_name=LEAGUE_NAME)
initial_year = loader.initial_sim_year
final_sim_year = loader.final_sim_year

# 3. Apply standard pipeline filtering (Ignore first year, apply right censoring)
valid_start_year = initial_year + 1
valid_end_year = final_sim_year - RIGHT_CENSOR_BUFFER

df = pl.read_parquet(FEATURES_PATH)
df_filtered = df.filter(
    (pl.col("Year") >= valid_start_year) & 
    (pl.col("Year") <= valid_end_year)
)

# 4. Perform Train/Test Split (Chronological)
total_valid_years = valid_end_year - valid_start_year + 1
test_years_count = int(total_valid_years * TEST_SPLIT_PCT)
train_end_year = valid_end_year - test_years_count

train_df = df_filtered.filter(pl.col("Year") <= train_end_year)

# 5. Extract DPO values > 0 (The "Hits")
dpo_values = train_df.filter(pl.col(TARGET_COL) > 10).get_column(TARGET_COL).to_numpy()

# 6. Plot Distribution
plt.figure(figsize=(10, 6))
sns.histplot(dpo_values, kde=True, color="#2ecc71", bins=30)
plt.title(f"Distribution of Positive {TARGET_COL} (Training Set: {valid_start_year}-{train_end_year})", fontsize=14)
plt.xlabel(f"{TARGET_COL} Value", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.grid(axis='y', alpha=0.3)
plt.show()

print(f"Total Training Samples: {len(train_df)}")
print(f"Positive Samples (Hits): {len(dpo_values)}")
print(f"Hit Rate: {len(dpo_values) / len(train_df):.2%}")
